In [0]:
%run ./01_shared_models_code_only

In [0]:
from typing import Any, Dict, List, Optional, TypedDict

In [0]:
class MultiAgentState(TypedDict):
    """
    Shared state passed between agents in the multi-agent workflow.
    """

    user_request: str

    coordinator_result: Optional[CoordinatorResult]

    agent_results: Dict[AgentName, BaseAgentResult]

    execution_history: List[ExecutionRecord]

    final_response: Optional[str]

    errors: List[Dict[str, Any]]

In [0]:
def create_initial_state(
    user_request: str,
) -> MultiAgentState:
    """
    Create the initial shared state for one customer request.

    Parameters
    ----------
    user_request:
        Original request submitted by the customer.

    Returns
    -------
    MultiAgentState
        New shared state containing empty workflow results.

    Raises
    ------
    ValueError
        If the request is empty or contains only whitespace.
    """

    cleaned_request = user_request.strip()

    if not cleaned_request:
        raise ValueError(
            "user_request must contain at least one "
            "non-whitespace character."
        )

    return {
        "user_request": cleaned_request,
        "coordinator_result": None,
        "agent_results": {},
        "execution_history": [],
        "final_response": None,
        "errors": [],
    }

In [0]:
def get_agent_task(
    tasks: List[AgentTask],
    agent_name: AgentName,
) -> Optional[AgentTask]:
    """
    Return the task assigned to a specified agent.

    Parameters
    ----------
    tasks:
        Execution-plan tasks created by the Coordinator Agent.

    agent_name:
        Specialist agent looking for its assigned task.

    Returns
    -------
    Optional[AgentTask]
        Assigned task when found; otherwise None.
    """

    for task in tasks:
        if task.agent_name == agent_name:
            return task

    return None

In [0]:
def record_agent_execution(
    state: MultiAgentState,
    agent_name: AgentName,
    status: AgentStatus,
    message: str,
) -> None:
    """
    Add one validated execution event to shared state.

    This function modifies the shared-state dictionary directly.
    Therefore, it does not return the state.

    Parameters
    ----------
    state:
        Current multi-agent workflow state.

    agent_name:
        Agent associated with the execution event.

    status:
        Execution status of the agent.

    message:
        Short description of the execution event.
    """

    execution_record = ExecutionRecord(
        agent_name=agent_name,
        status=status,
        message=message,
    )

    state["execution_history"].append(
        execution_record
    )

In [0]:

def record_agent_error(
    state: MultiAgentState,
    agent_name: AgentName,
    error_code: str,
    error_message: str,
) -> None:
    """
    Add one validated agent or workflow error to shared state.

    This function modifies the shared-state dictionary directly.
    Therefore, it does not return the state.

    Parameters
    ----------
    state:
        Current multi-agent workflow state.

    agent_name:
        Agent associated with the error.

    error_code:
        Standardized machine-readable error code.

    error_message:
        Human-readable description of the error.
    """

    error_record = AgentErrorRecord(
        agent_name=agent_name,
        error_code=error_code,
        error_message=error_message,
    )

    state["errors"].append(
        error_record
    )

In [0]:
def test_shared_state_and_helpers() -> None:
    """
    Run independent validation tests for shared-state
    initialization and reusable helper functions.
    """

    test_state = create_initial_state(
        "Video streaming keeps buffering."
    )

    test_tasks = [
        AgentTask(
            task_id="task_1",
            agent_name="prediction_agent",
            task_description=(
                "Predict the support-ticket category."
            ),
        ),
        AgentTask(
            task_id="task_2",
            agent_name="final_response_agent",
            task_description=(
                "Generate the final customer-facing response."
            ),
            depends_on=["prediction_agent"],
        ),
    ]

    assigned_task = get_agent_task(
        tasks=test_tasks,
        agent_name="prediction_agent",
    )

    missing_task = get_agent_task(
        tasks=test_tasks,
        agent_name="retention_agent",
    )

    record_agent_execution(
        state=test_state,
        agent_name="prediction_agent",
        status="success",
        message="Prediction task completed successfully.",
    )

    record_agent_error(
        state=test_state,
        agent_name="prediction_agent",
        error_code="TEST_ERROR",
        error_message=(
            "Example error used to validate error recording."
        ),
    )

    assert (
        test_state["user_request"]
        == "Video streaming keeps buffering."
    )

    assert (
        test_state["coordinator_result"]
        is None
    )

    assert (
        test_state["agent_results"]
        == {}
    )

    assert assigned_task is not None

    assert (
        assigned_task.agent_name
        == "prediction_agent"
    )

    assert missing_task is None

    assert len(
        test_state["execution_history"]
    ) == 1

    assert (
        test_state["execution_history"][0].agent_name
        == "prediction_agent"
    )

    assert (
        test_state["execution_history"][0].status
        == "success"
    )

    assert len(
        test_state["errors"]
    ) == 1

    assert (
        test_state["errors"][0].error_code
        == "TEST_ERROR"
    )

    try:
        create_initial_state("   ")

    except ValueError:
        pass

    else:
        raise AssertionError(
            "An empty request should raise ValueError."
        )

    print(
        "All shared-state and helper tests passed."
    )